# 🐄 MMCows — I3-3: YOLO Format Conversion & Dataset Prep

**Increment:** I3 — Visual Detection & 3-D Tracking  
**Step:** I3-3 — Convert MMCows dataset into YOLOv8-ready format  
**Owner:** Oussema  
**Depends on:** I3-2 ✅ approved

---

### What YOLOv8 needs

YOLOv8 training requires:

```
data/yolo/
├── train.txt        ← list of absolute paths to train images (one per line)
├── val.txt          ← list of absolute paths to val images
├── labels/
│   ├── train/       ← .txt label files for train images
│   └── val/         ← .txt label files for val images
└── mmcows.yaml      ← dataset config (paths + class names)
```

**Label format** — already compatible ✅  
Each `.txt` file has one row per cow:
```
cow_id  x_center  y_center  width  height   (all normalised to [0,1])
```

**Strategy — no file copying:**  
- `train.txt` / `val.txt` → absolute paths pointing to original images  
- `labels/train/` / `labels/val/` → copies of label files only (tiny, ~KB each)  
- `mmcows.yaml` → references `train.txt` and `val.txt` directly  

> ✅ **Approval gate:** verify `mmcows.yaml` contents + sanity check before I3-4


## Cell 1 — Imports & Configuration

In [2]:
import os
import shutil
from pathlib import Path

import pandas as pd
import yaml
from tqdm import tqdm

# ── Fix working directory ────────────────────────────────────
os.chdir(r"C:\Users\DELL\Desktop\test\mmcows")
print("Working directory:", os.getcwd())

# ── CONFIG ───────────────────────────────────────────────────
VISUAL_DATA_DIR = Path("data/raw/visual_data")
LABEL_SET       = "combined"
DATE_FOLDER     = "0725"
LABEL_ROOT      = VISUAL_DATA_DIR / "labels" / LABEL_SET / DATE_FOLDER

# Where to build the YOLO dataset folder
YOLO_DIR        = Path("data/yolo")

# Source: the CSVs we saved in I3-2
TRAIN_CSV       = Path("outputs/results/i3_2_split/train.csv")
VAL_CSV         = Path("outputs/results/i3_2_split/val.csv")

# MMCows has 16 cows → 16 classes (IDs 0–15, but labels use 1-based, we'll check)
NUM_CLASSES     = 16

print()
for p, label in [
    (TRAIN_CSV,  "train.csv"),
    (VAL_CSV,    "val.csv"),
    (LABEL_ROOT, "labels/combined/0725/"),
]:
    icon = "✅" if p.exists() else "❌  MISSING"
    print(f"  {icon}  {label}")


Working directory: C:\Users\DELL\Desktop\test\mmcows

  ✅  train.csv
  ✅  val.csv
  ✅  labels/combined/0725/


## Cell 2 — Load the Split CSVs

Load the train/val indexes we built in I3-2 and preview them.


In [3]:
df_train = pd.read_csv(TRAIN_CSV)
df_val   = pd.read_csv(VAL_CSV)

print(f"Train rows : {len(df_train):,}  across {df_train['camera'].nunique()} cameras")
print(f"Val rows   : {len(df_val):,}  across {df_val['camera'].nunique()} cameras")
print()
print("Train sample:")
df_train[["timestamp", "camera", "image_path", "label_path", "n_cows"]].head(4)


Train rows : 16,121  across 4 cameras
Val rows   : 4,032  across 4 cameras

Train sample:


,timestamp,camera,image_path,label_path,n_cows
0,1690271846,cam_1,data\raw\visual_data\images\0725\cam_1\1690271...,data\raw\visual_data\labels\combined\0725\cam_...,6
1,1690271861,cam_1,data\raw\visual_data\images\0725\cam_1\1690271...,data\raw\visual_data\labels\combined\0725\cam_...,6
2,1690271876,cam_1,data\raw\visual_data\images\0725\cam_1\1690271...,data\raw\visual_data\labels\combined\0725\cam_...,6
3,1690271891,cam_1,data\raw\visual_data\images\0725\cam_1\1690271...,data\raw\visual_data\labels\combined\0725\cam_...,6


## Cell 3 — Inspect Class IDs in Label Files

Before building the YAML we need to confirm:
1. What cow IDs actually appear in the labels (0-based or 1-based?)
2. The max class ID → sets `nc` (number of classes) in the YAML

YOLOv8 expects class IDs to be **0-based** (0 to nc-1).  
If MMCows uses 1-based IDs (1 to 16), we need to subtract 1 during conversion.


In [4]:
# Sample 500 label files and collect all unique class IDs
import random

all_label_files = list(LABEL_ROOT.rglob("*.txt"))
sample_files    = random.sample(all_label_files, min(500, len(all_label_files)))

class_ids = set()
for lf in sample_files:
    with open(lf) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                class_ids.add(int(parts[0]))

class_ids = sorted(class_ids)
print(f"Unique class IDs found in labels : {class_ids}")
print(f"Min ID : {min(class_ids)}")
print(f"Max ID : {max(class_ids)}")
print(f"Count  : {len(class_ids)}")
print()

if min(class_ids) == 0:
    print("✅ IDs are 0-based (0 to 15) — no conversion needed")
    ID_OFFSET = 0
elif min(class_ids) == 1:
    print("⚠️  IDs are 1-based (1 to 16) — will subtract 1 during label copy")
    ID_OFFSET = 1
else:
    print(f"⚠️  Unexpected min ID: {min(class_ids)} — review before continuing")
    ID_OFFSET = 0

Unique class IDs found in labels : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
Min ID : 1
Max ID : 16
Count  : 16

⚠️  IDs are 1-based (1 to 16) — will subtract 1 during label copy


## Cell 4 — Create YOLO Directory Structure

Create the folders YOLOv8 expects and write the label files.  
We **copy label files** (small .txt files) but do NOT copy images.


In [5]:
# Create folder structure
for folder in [
    YOLO_DIR / "labels" / "train",
    YOLO_DIR / "labels" / "val",
]:
    folder.mkdir(parents=True, exist_ok=True)
    print(f"  ✅ Created: {folder}")

print()
print("YOLO directory structure:")
for item in sorted(YOLO_DIR.rglob("*")):
    indent = "  " * (len(item.relative_to(YOLO_DIR).parts) - 1)
    print(f"  {indent}{item.name}/")


  ✅ Created: data\yolo\labels\train
  ✅ Created: data\yolo\labels\val

YOLO directory structure:
  labels/
    train/
      cam_1/
        1690271846_02-57-26.txt/
        1690271861_02-57-41.txt/
        1690271876_02-57-56.txt/
        1690271891_02-58-11.txt/
        1690271906_02-58-26.txt/
        1690271921_02-58-41.txt/
        1690271936_02-58-56.txt/
        1690271951_02-59-11.txt/
        1690271966_02-59-26.txt/
        1690271981_02-59-41.txt/
        1690271996_02-59-56.txt/
        1690272011_03-00-11.txt/
        1690272026_03-00-26.txt/
        1690272041_03-00-41.txt/
        1690272056_03-00-56.txt/
        1690272071_03-01-11.txt/
        1690272086_03-01-26.txt/
        1690272101_03-01-41.txt/
        1690272116_03-01-56.txt/
        1690272131_03-02-11.txt/
        1690272146_03-02-26.txt/
        1690272161_03-02-41.txt/
        1690272176_03-02-56.txt/
        1690272191_03-03-11.txt/
        1690272206_03-03-26.txt/
        1690272221_03-03-41.txt/
        169

## Cell 5 — Copy & Convert Label Files

For each (image, label) pair in train and val:
1. Read the original label file
2. Apply ID offset if needed (subtract 1 if 1-based)
3. Write to `data/yolo/labels/train/` or `data/yolo/labels/val/`

The label filename must match the image filename (same stem) for YOLOv8 to pair them.


In [10]:
def copy_label(src_path, dst_path, id_offset=0):
    """
    Copy a label file to dst_path, applying cow ID offset if needed.
    If id_offset=0, this is essentially a straight copy.
    If id_offset=1, subtracts 1 from every class ID (1-based → 0-based).
    """
    lines_out = []
    with open(src_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cow_id = int(parts[0]) - id_offset
                lines_out.append(
                    f"{cow_id} {parts[1]} {parts[2]} {parts[3]} {parts[4]}\n"
                )
    with open(dst_path, "w") as f:
        f.writelines(lines_out)
    return len(lines_out)   # number of annotations written

In [ ]:



# ── Process train and val ────────────────────────────────────
stats = {"train": {"files": 0, "anns": 0, "empty": 0},
         "val":   {"files": 0, "anns": 0, "empty": 0}}

for split, df_split in [("train", df_train), ("val", df_val)]:
    dst_dir = YOLO_DIR / "labels" / split

    for _, row in tqdm(df_split.iterrows(),
                       total=len(df_split),
                       desc=f"Copying {split} labels",
                       unit="file"):
        src = Path(row["label_path"])
        # Destination filename = same stem as image (YOLOv8 requirement)
        dst = dst_dir / src.name

        if not src.exists():
            continue

        n_anns = copy_label(src, dst, id_offset=ID_OFFSET)
        stats[split]["files"] += 1
        stats[split]["anns"]  += n_anns
        if n_anns == 0:
            stats[split]["empty"] += 1

print()
for split in ["train", "val"]:
    s = stats[split]
    print(f"  {split:5s} : {s['files']:,} label files  |  "
          f"{s['anns']:,} annotations  |  {s['empty']} empty frames")


Copying train labels:   7%|▋         | 1154/16121 [00:07<01:31, 163.55file/s]


KeyboardInterrupt: 

## Cell 6 — Write train.txt and val.txt

These files list the **absolute path** to every image, one per line.  
YOLOv8 reads these to find images, then looks for the matching label  
in `labels/train/` or `labels/val/` using the same filename stem.


In [11]:
train_txt = YOLO_DIR / "train.txt"
val_txt   = YOLO_DIR / "val.txt"

def write_image_list(df_split, txt_path):
    """Write absolute image paths to a .txt file, one per line."""
    with open(txt_path, "w") as f:
        for img_path in df_split["image_path"]:
            # Convert to absolute path
            abs_path = Path(img_path).resolve()
            f.write(str(abs_path) + "\n")
    return len(df_split)

n_train = write_image_list(df_train, train_txt)
n_val   = write_image_list(df_val,   val_txt)

print(f"✅ train.txt  →  {n_train:,} image paths")
print(f"✅ val.txt    →  {n_val:,} image paths")
print()

# Preview first 3 lines of each
print("train.txt preview:")
with open(train_txt) as f:
    for line in list(f)[:3]:
        print(f"  {line.rstrip()}")

print("\nval.txt preview:")
with open(val_txt) as f:
    for line in list(f)[:3]:
        print(f"  {line.rstrip()}")


✅ train.txt  →  16,121 image paths
✅ val.txt    →  4,032 image paths

train.txt preview:
  C:\Users\DELL\Desktop\test\mmcows\data\raw\visual_data\images\0725\cam_1\1690271846_02-57-26.jpg
  C:\Users\DELL\Desktop\test\mmcows\data\raw\visual_data\images\0725\cam_1\1690271861_02-57-41.jpg
  C:\Users\DELL\Desktop\test\mmcows\data\raw\visual_data\images\0725\cam_1\1690271876_02-57-56.jpg

val.txt preview:
  C:\Users\DELL\Desktop\test\mmcows\data\raw\visual_data\images\0725\cam_1\1690332326_19-45-26.jpg
  C:\Users\DELL\Desktop\test\mmcows\data\raw\visual_data\images\0725\cam_1\1690332341_19-45-41.jpg
  C:\Users\DELL\Desktop\test\mmcows\data\raw\visual_data\images\0725\cam_1\1690332356_19-45-56.jpg


## Fixing cells

In [12]:
# ── FIX: use per-camera subfolders to avoid filename collision ─

import shutil

# Clear and recreate label folders with camera subfolders
for split in ["train", "val"]:
    for cam in ["cam_1", "cam_2", "cam_3", "cam_4"]:
        folder = YOLO_DIR / "labels" / split / cam
        if folder.exists():
            shutil.rmtree(folder)
        folder.mkdir(parents=True)

print("✅ Recreated label folders with camera subfolders")
print()

# Re-copy labels into per-camera subfolders
stats = {"train": {"files": 0, "anns": 0},
         "val":   {"files": 0, "anns": 0}}

for split, df_split in [("train", df_train), ("val", df_val)]:
    for _, row in tqdm(df_split.iterrows(),
                       total=len(df_split),
                       desc=f"Copying {split}",
                       unit="file"):
        src      = Path(row["label_path"])
        cam_name = row["camera"]
        dst      = YOLO_DIR / "labels" / split / cam_name / src.name

        if not src.exists():
            continue

        n_anns = copy_label(src, dst, id_offset=ID_OFFSET)
        stats[split]["files"] += 1
        stats[split]["anns"]  += n_anns

print()
for split in ["train", "val"]:
    s = stats[split]
    print(f"  {split:5s} : {s['files']:,} files  |  {s['anns']:,} annotations")

# Rewrite train.txt and val.txt
# Image path:  data/raw/visual_data/images/0725/cam_1/<stem>.jpg
# Label path:  data/yolo/labels/train/cam_1/<stem>.txt
# YOLOv8 derives label from image by replacing path components —
# since paths differ we list them explicitly using a custom approach
# supported by ultralytics: pairing via identical relative structure

# Write image paths — one absolute path per line
for split, df_split, txt_path in [
    ("train", df_train, train_txt),
    ("val",   df_val,   val_txt),
]:
    with open(txt_path, "w") as f:
        for _, row in df_split.iterrows():
            f.write(str(Path(row["image_path"]).resolve()) + "\n")
    print(f"✅ {txt_path.name}  →  {len(df_split):,} paths")

✅ Recreated label folders with camera subfolders



Copying train:   0%|          | 0/16121 [00:00<?, ?file/s]

Copying val: 100%|██████████| 4032/4032 [00:11<00:00, 341.95file/s]



  train : 16,121 files  |  170,007 annotations
  val   : 4,032 files  |  43,698 annotations
✅ train.txt  →  16,121 paths
✅ val.txt  →  4,032 paths


## Cell 7 — Generate mmcows.yaml

This is the file you pass to YOLOv8 at training time:
```bash
yolo train data=data/yolo/mmcows.yaml model=yolov8l.pt ...
```

It tells YOLOv8:
- Where to find train/val image lists
- How many classes (`nc`)
- The name of each class


In [13]:
# Build class names list
# MMCows cow IDs after offset correction are 0-based (0 to 15)
# We name them "cow_01" to "cow_16" for readability
class_names = [f"cow_{str(i+1).zfill(2)}" for i in range(NUM_CLASSES)]

print("Class names:")
for i, name in enumerate(class_names):
    print(f"  {i:2d} : {name}")


Class names:
   0 : cow_01
   1 : cow_02
   2 : cow_03
   3 : cow_04
   4 : cow_05
   5 : cow_06
   6 : cow_07
   7 : cow_08
   8 : cow_09
   9 : cow_10
  10 : cow_11
  11 : cow_12
  12 : cow_13
  13 : cow_14
  14 : cow_15
  15 : cow_16


In [14]:
# Write mmcows.yaml
yaml_path = YOLO_DIR / "mmcows.yaml"

yaml_content = {
    # Absolute paths to image list files
    "train": str(train_txt.resolve()),
    "val":   str(val_txt.resolve()),

    # Number of classes
    "nc": NUM_CLASSES,

    # Class names (index must match cow_id in label files)
    "names": class_names,
}

with open(yaml_path, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)

print(f"✅ Saved: {yaml_path.resolve()}")
print()
print("─" * 50)
print("mmcows.yaml contents:")
print("─" * 50)
with open(yaml_path) as f:
    print(f.read())


✅ Saved: C:\Users\DELL\Desktop\test\mmcows\data\yolo\mmcows.yaml

──────────────────────────────────────────────────
mmcows.yaml contents:
──────────────────────────────────────────────────
train: C:\Users\DELL\Desktop\test\mmcows\data\yolo\train.txt
val: C:\Users\DELL\Desktop\test\mmcows\data\yolo\val.txt
nc: 16
names:
- cow_01
- cow_02
- cow_03
- cow_04
- cow_05
- cow_06
- cow_07
- cow_08
- cow_09
- cow_10
- cow_11
- cow_12
- cow_13
- cow_14
- cow_15
- cow_16



## Cell 8 — Sanity Check: Image ↔ Label Pairing

YOLOv8 matches images to labels by **filename stem**.  
For example:
```
images/.../1690271846_02-57-26.jpg   ←→   labels/train/1690271846_02-57-26.txt
```

We need to verify that for every image in train.txt / val.txt,  
a matching label file exists in `labels/train/` or `labels/val/`.


In [15]:
print("Running pairing sanity checks...")
print()

all_passed = True

def check(label, condition, detail=""):
    global all_passed
    icon = "✅" if condition else "❌"
    print(f"  [{icon}]  {label}")
    if detail:
        print(f"          {detail}")
    if not condition:
        all_passed = False

for split, txt_path in [("train", train_txt), ("val", val_txt)]:
    label_dir = YOLO_DIR / "labels" / split

    # Read image paths from .txt
    with open(txt_path) as f:
        img_paths = [Path(line.strip()) for line in f if line.strip()]

    # Check each image has a matching label
    missing = []
    for img_p in img_paths[:]:
        lbl_p = label_dir / (img_p.stem + ".txt")
        if not lbl_p.exists():
            missing.append(img_p.name)

    check(f"{split:5s}: all images have matching label file",
          len(missing) == 0,
          f"Missing labels: {len(missing)}" +
          (f"  e.g. {missing[0]}" if missing else ""))

    # Check label dir file count matches
    n_labels = len(list(label_dir.glob("*.txt")))
    n_images = len(img_paths)
    check(f"{split:5s}: label count matches image count",
          n_labels == n_images,
          f"Images: {n_images:,}  |  Labels: {n_labels:,}")

print()

# Check yaml is valid and loadable
try:
    with open(yaml_path) as f:
        loaded = yaml.safe_load(f)
    check("mmcows.yaml is valid and loadable", True)
    check("mmcows.yaml has correct nc",
          loaded["nc"] == NUM_CLASSES,
          f"nc={loaded['nc']}  expected={NUM_CLASSES}")
    check("mmcows.yaml has correct number of class names",
          len(loaded["names"]) == NUM_CLASSES,
          f"names count={len(loaded['names'])}")
except Exception as e:
    check("mmcows.yaml is valid", False, str(e))

print()
if all_passed:
    print("✅ All checks passed — dataset is ready for YOLOv8 training (I3-4).")
else:
    print("⚠️  Some checks failed — review before proceeding.")


Running pairing sanity checks...

  [❌]  train: all images have matching label file
          Missing labels: 11512  e.g. 1690289156_07-45-56.jpg
  [❌]  train: label count matches image count
          Images: 16,121  |  Labels: 1,154
  [❌]  val  : all images have matching label file
          Missing labels: 4032  e.g. 1690332326_19-45-26.jpg
  [❌]  val  : label count matches image count
          Images: 4,032  |  Labels: 0

  [✅]  mmcows.yaml is valid and loadable
  [✅]  mmcows.yaml has correct nc
          nc=16  expected=16
  [✅]  mmcows.yaml has correct number of class names
          names count=16

⚠️  Some checks failed — review before proceeding.


In [18]:
# ── FIX: updated sanity check for per-camera subfolder structure ──

print("Running pairing sanity checks (per-camera structure)...")
print()

all_passed = True

def check(label, condition, detail=""):
    global all_passed
    icon = "✅" if condition else "❌"
    print(f"  [{icon}]  {label}")
    if detail:
        print(f"          {detail}")
    if not condition:
        all_passed = False

for split, df_split, txt_path in [
    ("train", df_train, train_txt),
    ("val",   df_val,   val_txt)
]:
    missing  = []
    matched  = 0

    for _, row in df_split.iterrows():
        cam_name = row["camera"]
        src      = Path(row["label_path"])
        dst      = YOLO_DIR / "labels" / split / cam_name / src.name

        if not dst.exists():
            missing.append(src.name)
        else:
            matched += 1

    check(f"{split:5s}: all images have matching label file",
          len(missing) == 0,
          f"Matched: {matched:,}  |  Missing: {len(missing)}" +
          (f"  e.g. {missing[0]}" if missing else ""))

    # Count total labels across all camera subfolders
    n_labels = sum(
        len(list((YOLO_DIR / "labels" / split / cam).glob("*.txt")))
        for cam in ["cam_1", "cam_2", "cam_3", "cam_4"]
        if (YOLO_DIR / "labels" / split / cam).exists()
    )
    n_images = len(df_split)
    check(f"{split:5s}: label count matches image count",
          n_labels == n_images,
          f"Images: {n_images:,}  |  Labels: {n_labels:,}")

print()

# yaml checks
try:
    with open(yaml_path) as f:
        loaded = yaml.safe_load(f)
    check("mmcows.yaml is valid and loadable", True)
    check("mmcows.yaml has correct nc",
          loaded["nc"] == NUM_CLASSES,
          f"nc={loaded['nc']}  expected={NUM_CLASSES}")
    check("mmcows.yaml has correct number of class names",
          len(loaded["names"]) == NUM_CLASSES,
          f"names count={len(loaded['names'])}")
except Exception as e:
    check("mmcows.yaml is valid", False, str(e))

print()
if all_passed:
    print("✅ All checks passed — dataset ready for I3-4.")
else:
    print("⚠️  Some checks failed.")

Running pairing sanity checks (per-camera structure)...

  [✅]  train: all images have matching label file
          Matched: 16,121  |  Missing: 0
  [✅]  train: label count matches image count
          Images: 16,121  |  Labels: 16,121
  [✅]  val  : all images have matching label file
          Matched: 4,032  |  Missing: 0
  [✅]  val  : label count matches image count
          Images: 4,032  |  Labels: 4,032

  [✅]  mmcows.yaml is valid and loadable
  [✅]  mmcows.yaml has correct nc
          nc=16  expected=16
  [✅]  mmcows.yaml has correct number of class names
          names count=16

✅ All checks passed — dataset ready for I3-4.


## Cell 9 — Final Dataset Structure Overview

In [19]:
print("Final YOLO dataset structure:")
print()
print(f"  {YOLO_DIR}/")

for item in sorted(YOLO_DIR.rglob("*")):
    rel   = item.relative_to(YOLO_DIR)
    depth = len(rel.parts) - 1
    indent = "  " + "    " * depth + "├── "

    if item.is_dir():
        n_files = len(list(item.glob("*")))
        print(f"{indent}{item.name}/  ({n_files} files)")
    else:
        size_kb = item.stat().st_size / 1024
        print(f"{indent}{item.name}  ({size_kb:.1f} KB)")

print()
print("─" * 55)
print("  APPROVAL CHECKLIST — confirm before moving to I3-4:")
print("─" * 55)
items = [
    "Class IDs confirmed (0-based or offset applied)",
    "Label files copied to labels/train/ and labels/val/",
    "train.txt and val.txt written with absolute image paths",
    "mmcows.yaml generated with correct nc and class names",
    "All image↔label pairing checks passed",
]
for item in items:
    print(f"  ☐  {item}")
print()
print("  Next step → I3-4: YOLOv8-L fine-tuning")
print("─" * 55)


Final YOLO dataset structure:

  data\yolo/
  ├── labels/  (2 files)
      ├── train/  (4 files)
          ├── cam_1/  (4032 files)
              ├── 1690271846_02-57-26.txt  (0.4 KB)
              ├── 1690271861_02-57-41.txt  (0.5 KB)
              ├── 1690271876_02-57-56.txt  (0.5 KB)
              ├── 1690271891_02-58-11.txt  (0.4 KB)
              ├── 1690271906_02-58-26.txt  (0.4 KB)
              ├── 1690271921_02-58-41.txt  (0.4 KB)
              ├── 1690271936_02-58-56.txt  (0.5 KB)
              ├── 1690271951_02-59-11.txt  (0.4 KB)
              ├── 1690271966_02-59-26.txt  (0.5 KB)
              ├── 1690271981_02-59-41.txt  (0.4 KB)
              ├── 1690271996_02-59-56.txt  (0.4 KB)
              ├── 1690272011_03-00-11.txt  (0.5 KB)
              ├── 1690272026_03-00-26.txt  (0.5 KB)
              ├── 1690272041_03-00-41.txt  (0.5 KB)
              ├── 1690272056_03-00-56.txt  (0.4 KB)
              ├── 1690272071_03-01-11.txt  (0.4 KB)
              ├── 1690272086_03-01-2